# IST-3

This notebook evaluates whether TFMs can improve uplift modeling performance when used inside meta-learners compared to conventional baseline methods and a standalone causal model (CausalPFN).

**Dataset**: International Stroke Trial 3 (IST-3) - a randomised controlled trial of intravenous rt-PA vs. placebo in acute ischaemic stroke.
- 2752 samples (after dropping rows with missing covariates), 15 covariates, binary outcome = alive and independent at 6 months (Y = 1 if aliveind6 = yes)
- Treatment: `itt_treat` recoded so T=1 = rt-PA, T=0 = placebo (~50/50 split)
- Covariates: age, weight, glucose, GCS score, NIHSS, SBP, DBP (continuous); gender, antiplatelet use, atrial fibrillation, infarct present (binary); stroke type (one-hot encoded)

**Evaluation protocol**:
- RepeatedStratifiedKFold: 5 folds × 4 repeats = 20 splits (test size ≈ 20%), stratified on T × Y
- All models trained on training set only, evaluated on held-out test set
- R-learner (NonParamDML) and DR-learner (DRLearner) use 5-fold cross-fitting within the training data for nuisance estimation
- LightGBM hyperparameters tuned once per split via GridSearchCV (27 combinations, 3-fold CV)

**Metrics** (all computed on the test set per split):
- **AUQC**: Area between the Qini curve and its diagonal (the random-model baseline). Expected value under a random CATE ranking is 0 by construction; significance test checks whether the 95% CI excludes zero.
- **Policy Gain @k%** (k = 10, 20, 30, 40): IPW policy value for treating the top-k% minus the random k% baseline — already measures improvement over random targeting

**Aggregation**: Mean ± SE are computed over all K×R = 20 fold-level estimates using the Nadeau–Bengio variance correction (Bouckaert & Frank 2004): SE = sqrt((1/(K×R) + n₂/n₁) × σ̂²), where K=5, R=4, n₂/n₁=0.25 (test/train ratio for 5-fold CV), and σ̂² is the sample variance across all 20 folds. The 95% CI and significance test use a t-distribution with df = 19 (K×R − 1). Since Y = alive/independent is a positive outcome, significance is one-sided: `*` when ci_lo > 0.

**Models**:
- Meta-learners: S, T, X, R (NonParamDML, cv=5), DR (DRLearner, cv=5)
- Base models: LinearRegression, LightGBM (tuned), TabPFN v2.5, TabPFN v2.6, TabICL
- Standalone: CausalForestDML (cv=5, LightGBM nuisance), CausalPFN

## 1. Setup and Imports

In [ ]:
import os
os.environ["TABPFN_NO_TELEMETRY_PROMPT"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# ── IMPORTS ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import scipy.stats as st
import pickle
from collections import defaultdict
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold, RepeatedStratifiedKFold
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
import tabpfn
from tabpfn import TabPFNRegressor, TabPFNClassifier
from tabicl import TabICLRegressor, TabICLClassifier
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import NonParamDML, CausalForestDML
from econml.dr import DRLearner
import matplotlib.pyplot as plt
import warnings
import torch

from causalpfn import CATEEstimator
from causalpfn.evaluation import get_qini_curve as _get_qini_curve

def _compute_auqc(qini_curve):
    n = len(qini_curve)
    phi = np.linspace(1/n, 1.0, n)
    diagonal = phi * qini_curve[-1]
    return np.trapezoid(qini_curve - diagonal, phi)

def get_qini_curve(T, Y, cate):
    # _compute_auqc always computes the diagonal-subtracted area
    curve, _ = _get_qini_curve(T, Y, cate, normalize=False)
    return curve, _compute_auqc(curve)


# ── DEVICE DETECTION ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"TabPFN {tabpfn.__version__} | device: {device}")

# CausalPFN does not support MPS
causalpfn_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"CausalPFN device: {causalpfn_device}")

# TabICL does not support MPS — fall back to CPU
tabicl_device = "cpu" if device == "mps" else device
print(f"TabICL device: {tabicl_device}")


# ── GLOBAL SEED ──────────────────────────────────────────────────────────────
SEED = 42

# ── EXTRA ─────────────────────────────────────────────────────────
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  # no-op if no CUDA
warnings.filterwarnings("ignore")

# Results output directory — timestamped to avoid overwriting previous runs
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = os.path.join('results', f'exp_05_IST3_{RUN_ID}')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results directory: {RESULTS_DIR}")

In [3]:
# ── Import TabPFN 2.5 alongside 2.6 ────────────────────────────────────────
import subprocess, sys as _sys

TABPFN25_DIR = "./tabpfn_v25_install"
os.makedirs(TABPFN25_DIR, exist_ok=True)

# Install tabpfn 2.5 to isolated directory if not already present
_tabpfn25_installed = any("tabpfn" in d for d in os.listdir(TABPFN25_DIR))
if not _tabpfn25_installed:
    print("Installing TabPFN 2.5 to isolated directory...")
    subprocess.check_call([
        _sys.executable, "-m", "pip", "install", "tabpfn==6.4.1",
        f"--target={TABPFN25_DIR}", "--quiet", "--no-deps"
    ])
    print("Done.")

# Save current tabpfn 2.6 module references
_tabpfn26_mods = {k: v for k, v in _sys.modules.items()
                  if k == "tabpfn" or k.startswith("tabpfn.")}

# Temporarily inject 2.5 path and load its classes
_sys.path.insert(0, TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]

import tabpfn as _tabpfn25
TabPFNRegressor25 = _tabpfn25.TabPFNRegressor
TabPFNClassifier25 = _tabpfn25.TabPFNClassifier
TABPFN25_VERSION = _tabpfn25.__version__

# Restore tabpfn 2.6
_sys.path.remove(TABPFN25_DIR)
for _k in [k for k in list(_sys.modules) if k == "tabpfn" or k.startswith("tabpfn.")]:
    del _sys.modules[_k]
_sys.modules.update(_tabpfn26_mods)

print(f"TabPFN 2.5 version: {TABPFN25_VERSION}")
print(f"TabPFN 2.6 version: {tabpfn.__version__}")

Installing TabPFN 2.5 to isolated directory...
Done.
TabPFN 2.5 version: 6.4.1
TabPFN 2.6 version: 7.1.1


## 2. Helper Functions

LightGBM tuning via `GridSearchCV` (27 combinations, 3-fold CV, 3 hyperparameters). IPW-based policy value estimation for top-k% targeting. Qini curve interpolation. AUQC is computed as the area between the Qini curve and its diagonal (the random-model baseline), so its expected value under a random CATE ranking is 0 by construction.

In [ ]:
# ── TUNING ────────────────────────────────────────────────────
# LightGBM hyperparameter search space
LGBM_GRID = {
    'num_leaves':        [15, 31, 63],
    'min_child_samples': [20, 50, 100],
    #'learning_rate':     [0.01, 0.05, 0.1],
    'n_estimators':      [200, 500, 1000],
}
# 3×3×3 = 27 combinations — full grid

NUISANCE_CV   = 5    # K-fold cross-fitting for R- and DR-learner nuisance models

# LightGBM Tuning
def tune_lgbm(X, y, classifier=False, stratify=None, seed=SEED):
    """Tune LGBM via GridSearchCV. Returns best_params_ dict.

    - classifier=False  → LGBMRegressor, scored by neg_MSE
    - classifier=True   → LGBMClassifier, scored by neg_log_loss
    - stratify          → use StratifiedKFold on this variable (regression only);
                          if None, falls back to KFold with adaptive n_splits
    """
    X = np.asarray(X)

    if classifier:
        # Propensity Model (classification)
        base    = LGBMClassifier(random_state=seed, verbose=-1)
        scoring = 'neg_log_loss'
        cv      = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y))
    else:
        # Outcome Model (regression): S-, R-, and DR-learner
        base    = LGBMRegressor(random_state=seed, verbose=-1)
        scoring = 'neg_mean_squared_error'
        if stratify is not None:
            cv  = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, stratify))
        else:
            # Single Treatment Arm: T- and X-learner
            n_splits = min(3, max(2, len(y) // 20))
            cv  = list(KFold(n_splits, shuffle=True, random_state=seed).split(X))

    search = GridSearchCV(
        base, LGBM_GRID, scoring=scoring,
        cv=cv, n_jobs=-1,
    )

    search.fit(X, y)
    return search.best_params_


# Pseudo-Outcome Tuning: X-, R-, and DR-learner final-stage models
def make_lgbm_final(seed=SEED):
    """LGBM wrapped in GridSearchCV for final-stage tuning on pseudo-outcomes."""
    return GridSearchCV(
        LGBMRegressor(random_state=seed, verbose=-1),
        LGBM_GRID, cv=3, scoring='neg_mean_squared_error',
        n_jobs=-1,
    )


# ── Metrics ───────────────────────────────────────────────────
TOP_K_PERCENTAGES = [10, 20, 30, 40]

def calculate_topk_policy_metrics(y_true, treatment, predicted_ite,
                                   k_percentages=TOP_K_PERCENTAGES, min_treated=3, min_control=3):
    """IPW policy value for top-k% treatment policies (unbiased under RCT design)."""
    treatment = np.asarray(treatment)
    y_true = np.asarray(y_true)
    predicted_ite = np.asarray(predicted_ite).flatten()
    n = len(y_true)
    sorted_idx = np.argsort(-predicted_ite)
    e = treatment.mean()
    treat_all = (treatment * y_true / e).mean() if e > 0 else np.nan
    treat_none = ((1 - treatment) * y_true / (1 - e)).mean() if e < 1 else np.nan
    metrics = {}
    for k in k_percentages:
        n_top = int(np.ceil(n * k / 100))
        pi = np.zeros(n)
        pi[sorted_idx[:n_top]] = 1.0
        n_t_top = (treatment[sorted_idx[:n_top]] == 1).sum()
        n_c_bot = (treatment[sorted_idx[n_top:]] == 0).sum()
        if n_t_top < min_treated or n_c_bot < min_control:
            pv = np.nan
        else:
            pv = (pi * treatment * y_true / e + (1 - pi) * (1 - treatment) * y_true / (1 - e)).mean()
        rand_k = (k / 100) * treat_all + (1 - k / 100) * treat_none if not (np.isnan(treat_all) or np.isnan(treat_none)) else np.nan
        gain = pv - rand_k if not (np.isnan(pv) or np.isnan(rand_k)) else np.nan
        metrics[f'policy_value_{k}'] = pv
        metrics[f'random_policy_{k}'] = rand_k
        metrics[f'policy_gain_{k}'] = gain
    return metrics

def interpolate_qini_curve(qini_curve, n_grid=101):
    n = len(qini_curve)
    x_orig = np.concatenate([[0.0], np.linspace(1/n, 1.0, n)])
    y_orig = np.concatenate([[0.0], qini_curve])
    return np.interp(np.linspace(0.0, 1.0, n_grid), x_orig, y_orig)

## 3. Data Loading

Load the IST-3 dataset from a local SAS file and define stratification variables and run configuration.

In [ ]:
# --- Load IST-3 data ---
DATA_DIR  = os.path.join(os.path.dirname(os.path.abspath('IST3.ipynb')), 'data')
SAS_PATH  = os.path.join(DATA_DIR, 'datashare_aug2015.sas7bdat')

raw = pd.read_sas(SAS_PATH, encoding='latin1')

# ── Treatment ─────────────────────────────────────────────────────────────────
# itt_treat: 0 = rt-PA, 1 = placebo  →  recode so T=1 means treated (rt-PA)
raw['T'] = (raw['itt_treat'] == 0).astype(int)

# ── Outcome ───────────────────────────────────────────────────────────────────
# aliveind6: 1 = alive and independent, 2 = otherwise  →  Y=1 if alive/independent
raw['Y'] = (raw['aliveind6'] == 1.0).astype(float)

# ── Covariates ────────────────────────────────────────────────────────────────
# Continuous
continuous_cols = ['age', 'weight', 'glucose', 'gcs_score_rand', 'nihss', 'sbprand', 'dbprand']

# Binary: SAS encoding 1=yes/male, 2=no/female  →  recode to 0/1
binary_cols = ['gender', 'antiplat_rand', 'atrialfib_rand']
for col in binary_cols:
    raw[col] = (raw[col] == 1.0).astype(int)

# infarct: 0=no infarct, 1/2=infarct present  →  binary present/absent
raw['infarct'] = (raw['infarct'] > 0).astype(int)

# stroketype: one-hot encode (drop first to avoid multicollinearity)
stroketype_dummies = pd.get_dummies(raw['stroketype'], prefix='stroketype', drop_first=True).astype(int)

# ── Assemble X, T, Y ─────────────────────────────────────────────────────────
feature_cols = continuous_cols + binary_cols + ['infarct'] + list(stroketype_dummies.columns)
X_raw = pd.concat([raw[continuous_cols + binary_cols + ['infarct']], stroketype_dummies], axis=1)

df_ist3 = pd.concat([X_raw, raw[['T', 'Y']]], axis=1).dropna().reset_index(drop=True)

T_all = df_ist3['T'].values.astype(int)
Y_all = df_ist3['Y'].values.astype(float)
X_all = df_ist3[feature_cols].reset_index(drop=True)

# ── Stratification variable: interaction of treatment and outcome ──────────────
strat_all    = T_all * 2 + Y_all.astype(int)
strat_labels = {0: 'control+not-alive/indep', 1: 'control+alive/indep',
                2: 'treated+not-alive/indep', 3: 'treated+alive/indep'}

# ── Data summary ──────────────────────────────────────────────────────────────
n_total   = len(X_all)
n_treated = int(T_all.sum())
n_control = n_total - n_treated
n_outcome = int(Y_all.sum())
ate       = Y_all[T_all == 1].mean() - Y_all[T_all == 0].mean()

print("=" * 60)
print("IST-3 Dataset Summary")
print("=" * 60)
print(f"  Total samples  : {n_total}")
print(f"  Treated (rt-PA): {n_treated}  ({100*n_treated/n_total:.1f}%)")
print(f"  Control(placebo): {n_control}  ({100*n_control/n_total:.1f}%)")
print(f"  Features ({len(feature_cols):2d})  : {feature_cols}")
print(f"  Outcome (Y)    : {n_outcome} alive/independent ({100*n_outcome/n_total:.1f}%) [Y=1=alive/indep]")
print(f"  ATE (naive)    : {ate:+.4f}  (treated mean - control mean)")
print()
print("Outcome by treatment arm:")
print(f"  Treated  alive/indep rate: {Y_all[T_all==1].mean():.4f}")
print(f"  Control  alive/indep rate: {Y_all[T_all==0].mean():.4f}")
print()
print("Stratification strata counts (T x Y):")
for val, label in strat_labels.items():
    count = (strat_all == val).sum()
    min_needed = int(np.ceil(count * 0.2))
    print(f"  {label:<30}: {count:3d} total, ~{min_needed:2d} in test  "
          f"{'OK' if count >= 10 else 'WARNING: very small'}")

# ── Configuration ─────────────────────────────────────────────────────────────
DRY_RUN = False

N_FOLDS   = 2  if DRY_RUN else 5
N_REPEATS = 1  if DRY_RUN else 4
N_SPLITS  = N_FOLDS * N_REPEATS   # 5 × 4 = 20

N_GRID   = 101

# ── Storage ───────────────────────────────────────────────────────────────────
results     = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
qini_curves = defaultdict(lambda: defaultdict(list))
# Track which repeat each split belongs to (for repeat-aware SE)
split_repeat_idx = []

META_LEARNERS = ['S', 'T', 'X', 'R', 'DR', 'CF', 'CausalPFN']
BASE_MODELS = ['LinearRegression', 'LightGBM', 'TabPFN_v2.5','TabPFN_v2.6', 'TabICL']

print(f"\nConfiguration: {N_FOLDS} folds × {N_REPEATS} repeats = {N_SPLITS} splits (test ≈ 20%), DRY_RUN={DRY_RUN}")

## 4. Evaluation

Run all models across the repeated stratified splits. Per split we:
1. Tune LightGBM once (regressor + classifier), reuse across all meta-learners
2. Fit S/T/X/R/DR-learners with 5 base models + CausalForestDML + CausalPFN, predict on test set
3. Store AUQC (diagonal-subtracted) and policy gains for later aggregation

**Stratification**: splits are stratified on `T × Y` — a 4-category interaction of treatment and binary outcome — to ensure each split preserves the failure rate within each treatment arm.

In [ ]:
import time

rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED)
splits = list(rskf.split(X_all, strat_all))

print(f"Running {N_FOLDS} folds × {N_REPEATS} repeats = {len(splits)} splits...\n")

for split_idx, (train_idx, test_idx) in enumerate(splits):
    repeat_idx = split_idx // N_FOLDS
    fold_idx   = split_idx  % N_FOLDS
    split_repeat_idx.append(repeat_idx)

    X_train = X_all.iloc[train_idx].reset_index(drop=True)
    X_test  = X_all.iloc[test_idx].reset_index(drop=True)
    T_train, T_test = T_all[train_idx], T_all[test_idx]
    Y_train, Y_test = Y_all[train_idx], Y_all[test_idx]

    ate_test = Y_test[T_test == 1].mean() - Y_test[T_test == 0].mean()
    print(f"\n{'='*60}")
    print(f"Repeat {repeat_idx + 1}/{N_REPEATS}, Fold {fold_idx + 1}/{N_FOLDS}  "
          f"(split {split_idx + 1}/{N_SPLITS})  |  test ATE: {ate_test:+.4f}"
          f"  (n={len(T_test)}, treated={T_test.sum()}, control={(T_test==0).sum()})")
    print(f"{'='*60}")

    # ── Arm splits (needed for T- and X-learner tuning) ──────────────────────
    trt_mask  = T_train == 1
    ctrl_mask = T_train == 0
    X_trt, Y_trt = X_train[trt_mask].reset_index(drop=True), Y_train[trt_mask]
    X_ctrl, Y_ctrl = X_train[ctrl_mask].reset_index(drop=True), Y_train[ctrl_mask]


    # ── LightGBM hyperparameter tuning ───────────────────────────────────────────
    print(f"  [LightGBM] Tuning hyperparameters...", end=" ", flush=True)
    t0 = time.time()
    X_with_T        = np.column_stack([X_train, T_train])
    params_s        = tune_lgbm(X_with_T, Y_train, stratify=T_train)  # S-learner outcome (X+T features)
    params_outcome  = tune_lgbm(X_train,  Y_train, stratify=T_train)  # outcome nuisance (R/DR)
    params_prop     = tune_lgbm(X_train,  T_train, classifier=True)   # propensity model
    params_ctrl     = tune_lgbm(X_ctrl,   Y_ctrl)                     # T/X control arm outcome
    params_trt      = tune_lgbm(X_trt,    Y_trt)                      # T/X treated arm outcome
    print(f"  LightGBM tuning: {time.time() - t0:.1f}s")


    # ── Model configs ────────────────────────────────────────────────────────
    # Local shorthands to avoid repeating constructor args across 4 models × 11 roles
    def _lgbm_r(params):    return LGBMRegressor(random_state=SEED, verbose=-1, **params)
    def _lgbm_c(params):    return LGBMClassifier(random_state=SEED, verbose=-1, **params)
    def _tabpfn_r():        return TabPFNRegressor(device=device, random_state=SEED)
    def _tabpfn_c():        return TabPFNClassifier(device=device, random_state=SEED)
    def _tabicl_r():        return TabICLRegressor(device=tabicl_device, random_state=SEED, verbose=False)
    def _tabicl_c():        return TabICLClassifier(device=tabicl_device, random_state=SEED, verbose=False)
    def _tabpfn25_r():      return TabPFNRegressor25(device=device, random_state=SEED)
    def _tabpfn25_c():      return TabPFNClassifier25(device=device, random_state=SEED)

    configs = {
        'LinearRegression': {
            's_model':       LinearRegression(),
            't_models':      (LinearRegression(), LinearRegression()),
            'x_models':      (LinearRegression(), LinearRegression()),
            'x_cate':        (LinearRegression(), LinearRegression()),
            'x_propensity':  LogisticRegression(max_iter=1000, random_state=SEED),
            'r_model_y':     LinearRegression(),
            'r_model_t':     LogisticRegression(max_iter=1000, random_state=SEED),
            'r_model_final': LinearRegression(),
            'dr_regression': LinearRegression(),
            'dr_propensity': LogisticRegression(max_iter=1000, random_state=SEED),
            'dr_final':      LinearRegression(),
        },
        'LightGBM': {
            's_model':       _lgbm_r(params_s),
            # T/X-learner: (models[0]=control arm, models[1]=treated arm) — EconML convention
            't_models':      (_lgbm_r(params_ctrl), _lgbm_r(params_trt)),
            'x_models':      (_lgbm_r(params_ctrl), _lgbm_r(params_trt)),
            'x_cate':        (make_lgbm_final(), make_lgbm_final()),
            'x_propensity':  _lgbm_c(params_prop),
            'r_model_y':     _lgbm_r(params_outcome),
            'r_model_t':     _lgbm_c(params_prop),
            'r_model_final': make_lgbm_final(),
            'dr_regression': _lgbm_r(params_s),       # tuned on [X,T], matches DRLearner's internal input
            'dr_propensity': _lgbm_c(params_prop),
            'dr_final':      make_lgbm_final(),
        },
        'TabPFN_v2.5': {
            's_model':       _tabpfn25_r(),
            't_models':      (_tabpfn25_r(), _tabpfn25_r()),
            'x_models':      (_tabpfn25_r(), _tabpfn25_r()),
            'x_cate':        (_tabpfn25_r(), _tabpfn25_r()),
            'x_propensity':  _tabpfn25_c(),
            'r_model_y':     _tabpfn25_r(),
            'r_model_t':     _tabpfn25_c(),
            'r_model_final': make_lgbm_final(),
            'dr_regression': _tabpfn25_r(),
            'dr_propensity': _tabpfn25_c(),
            'dr_final':      _tabpfn25_r(),
        },
        'TabPFN_v2.6': {
            's_model':       _tabpfn_r(),
            't_models':      (_tabpfn_r(), _tabpfn_r()),
            'x_models':      (_tabpfn_r(), _tabpfn_r()),
            'x_cate':        (_tabpfn_r(), _tabpfn_r()),
            'x_propensity':  _tabpfn_c(),
            'r_model_y':     _tabpfn_r(),
            'r_model_t':     _tabpfn_c(),
            'r_model_final': make_lgbm_final(),  # TabPFN not suited for residual-on-residual
            'dr_regression': _tabpfn_r(),
            'dr_propensity': _tabpfn_c(),
            'dr_final':      _tabpfn_r(),
        },
        'TabICL': {
            's_model':       _tabicl_r(),
            # T/X-learner: (models[0]=control arm, models[1]=treated arm) — EconML convention
            't_models':      (_tabicl_r(), _tabicl_r()),
            'x_models':      (_tabicl_r(), _tabicl_r()),
            'x_cate':        (_tabicl_r(), _tabicl_r()),
            'x_propensity':  _tabicl_c(),
            'r_model_y':     _tabicl_r(),
            'r_model_t':     _tabicl_c(),
            'r_model_final': make_lgbm_final(),  # TabICL not suited for residual-on-residual
            'dr_regression': _tabicl_r(),
            'dr_propensity': _tabicl_c(),
            'dr_final':      _tabicl_r(),
        },
    }

    def evaluate(meta, name, te_pred):
        """Compute and store all metrics for one (meta, name, split).
        Y = aliveind6 (1=alive/independent): CATE > 0 means treatment helps.
        Pass te directly so Qini ranks highest-benefit individuals first.
        """
        te = np.asarray(te_pred).flatten()
        curve, auqc = get_qini_curve(T_test, Y_test, te)
        results[meta][name]['auqc'].append(auqc)
        results[meta][name]['g1'].append(float(curve[-1]))
        qini_curves[meta][name].append(interpolate_qini_curve(curve, n_grid=N_GRID))
        topk = calculate_topk_policy_metrics(Y_test, T_test, te)  # Y=1 is good outcome, positive gain = better
        for k in TOP_K_PERCENTAGES:
            results[meta][name][f'gain_{k}'].append(topk[f'policy_gain_{k}'])


    # ── Evaluation ────────────────────────────────────────────────────────────
    for name, cfg in configs.items():

        # ── S-Learner ───────────────────────────────────────────────────────────────
        print(f"  [{name}] S-learner...", end=" ", flush=True)
        _t0 = time.time()
        sl = SLearner(overall_model=cfg['s_model'])
        sl.fit(Y_train, T_train, X=X_train)
        evaluate('S', name, sl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── T-Learner ───────────────────────────────────────────────────────────────
        print(f"  [{name}] T-learner...", end=" ", flush=True)
        _t0 = time.time()
        tl = TLearner(models=cfg['t_models'])
        tl.fit(Y_train, T_train, X=X_train)
        evaluate('T', name, tl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── X-Learner ───────────────────────────────────────────────────────────────
        print(f"  [{name}] X-learner...", end=" ", flush=True)
        _t0 = time.time()
        xl = XLearner(models=cfg['x_models'], cate_models=cfg['x_cate'], propensity_model=cfg['x_propensity'])
        xl.fit(Y_train, T_train, X=X_train)
        evaluate('X', name, xl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── R-learner (NonParamDML) ──────────────────────────────────────────────
        print(f"  [{name}] R-learner (NonParamDML, cv={NUISANCE_CV})...", end=" ", flush=True)
        _t0 = time.time()
        rl = NonParamDML(
            model_y=cfg['r_model_y'], model_t=cfg['r_model_t'],
            model_final=cfg['r_model_final'], discrete_treatment=True, cv=NUISANCE_CV, random_state=SEED
        )
        rl.fit(Y_train, T_train, X=X_train)
        evaluate('R', name, rl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


        # ── DR-learner ───────────────────────────────────────────────────────────
        print(f"  [{name}] DR-learner (DRLearner, cv={NUISANCE_CV})...", end=" ", flush=True)
        _t0 = time.time()
        dl = DRLearner(
            model_regression=cfg['dr_regression'],
            model_propensity=cfg['dr_propensity'],
            model_final=cfg['dr_final'],
            min_propensity=0.05,
            cv=NUISANCE_CV,
            random_state=SEED
        )
        dl.fit(Y_train, T_train, X=X_train)
        evaluate('DR', name, dl.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")


    # ── Causal Forest (CausalForestDML) ───────────────────────────────────────
    try:
        print(f"  [CausalForest] DML (cv={NUISANCE_CV})...", end=" ", flush=True)
        _t0 = time.time()
        cf = CausalForestDML(
            model_y=LGBMRegressor(random_state=SEED, verbose=-1, **params_outcome),
            model_t=LGBMClassifier(random_state=SEED, verbose=-1, **params_prop),
            discrete_treatment=True,
            cv=NUISANCE_CV,
            n_estimators=200,
            min_samples_leaf=5,
            random_state=SEED,
        )
        cf.tune(Y_train, T_train, X=X_train)
        cf.fit(Y_train, T_train, X=X_train)
        evaluate('CF', 'CausalForest', cf.effect(X_test))
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalForest error on split {split_idx + 1}: {exc}")


    # ── CausalPFN ───────────────────────────────────────────────────────────────
    try:
        print(f"  [CausalPFN]...", end=" ", flush=True)
        _t0 = time.time()
        cpfn = CATEEstimator(device=causalpfn_device, verbose=False)

        X_cpfn_train = np.asarray(X_train, dtype=np.float32)
        T_cpfn_train = np.asarray(T_train, dtype=np.float32).ravel()
        Y_cpfn_train = np.asarray(Y_train, dtype=np.float32).ravel()
        X_cpfn_test  = np.asarray(X_test, dtype=np.float32)

        cpfn.fit(X_cpfn_train, T_cpfn_train, Y_cpfn_train)
        te = cpfn.estimate_cate(X_cpfn_test)

        if "torch" in str(type(te)):
            te = te.detach().cpu().numpy()

        te = np.asarray(te, dtype=np.float32).reshape(-1)
        evaluate('CausalPFN', 'CausalPFN', te)
        print(f"done. ({time.time() - _t0:.1f}s)")
    except Exception as exc:
        print(f"ERROR\n  CausalPFN error on split {split_idx + 1}: {exc}")

print(f"\nDone. {N_SPLITS} splits evaluated.")

## Save Results to Disk

In [ ]:
# --- Save results to disk ---
_save_path = os.path.join(RESULTS_DIR, 'ist3_results.pkl')
with open(_save_path, 'wb') as f:
    pickle.dump({
        'results': {m: {n: dict(d) for n, d in v.items()} for m, v in results.items()},
        'qini_curves': {m: {n: list(c) for n, c in v.items()} for m, v in qini_curves.items()},
        'N_SPLITS': N_SPLITS,
        'N_FOLDS': N_FOLDS,
        'N_REPEATS': N_REPEATS,
        'N_GRID': N_GRID,
        'META_LEARNERS': META_LEARNERS,
        'BASE_MODELS': BASE_MODELS,
        'TOP_K_PERCENTAGES': TOP_K_PERCENTAGES,
        'split_repeat_idx': split_repeat_idx,
    }, f)
print(f"Results saved to {_save_path}")

## 5. Results

We aggregate per-split metrics across the 20 splits and report mean ± SE.

**Reported metrics**:
- AUQC (area between Qini curve and its diagonal): mean ± SE, 95% CI, significant if CI excludes zero. Expected value under a random CATE ranking is 0 by construction.
- Gain@10/20/30/40%: mean ± SE (already a delta: policy value minus random targeting baseline)

**Aggregation**: Mean is computed over all K×R = 20 fold-level estimates directly. The standard error uses the Nadeau–Bengio variance correction (Bouckaert & Frank 2004), which inflates the naive cross-fold SE to account for partial training-set overlap between folds:

$$\text{SE} = \sqrt{\left(\frac{1}{KR} + \frac{n_2}{n_1}\right) \hat{\sigma}^2}, \quad t = \frac{\bar{x}}{\text{SE}}, \quad df = KR - 1$$

where K = 5 folds, R = 4 repeats, n₂/n₁ = 0.25 (test/train ratio for 5-fold CV), and σ̂² is the sample variance across all 20 fold scores. The 95% CI and significance test use a t-distribution with df = 19. Since Y = alive/independent is a positive outcome, significance is one-sided: `*` when ci_lo > 0.

In [ ]:
# --- Load results from disk (run this cell instead of the evaluation loop when working locally) ---
# with open('<path-to>/ist3_results.pkl', 'rb') as f:
#     _data = pickle.load(f)
# results = defaultdict(lambda: defaultdict(lambda: defaultdict(list)), {
#     m: defaultdict(lambda: defaultdict(list), {n: defaultdict(list, d) for n, d in v.items()})
#     for m, v in _data['results'].items()
# })
# qini_curves = defaultdict(lambda: defaultdict(list), {
#     m: defaultdict(list, v) for m, v in _data['qini_curves'].items()
# })
# N_SPLITS          = _data['N_SPLITS']
# N_FOLDS           = _data['N_FOLDS']
# N_REPEATS         = _data['N_REPEATS']
# N_GRID            = _data['N_GRID']
# META_LEARNERS     = _data['META_LEARNERS']
# BASE_MODELS       = _data['BASE_MODELS']
# TOP_K_PERCENTAGES = _data['TOP_K_PERCENTAGES']
# split_repeat_idx  = _data['split_repeat_idx']
# print(f"Loaded results: {N_FOLDS} folds × {N_REPEATS} repeats = {N_SPLITS} splits, {len(META_LEARNERS)} meta-learners")

In [ ]:
# --- Aggregate ---
# Bouckaert & Frank (2004) corrected SE for repeated K-fold CV.
# Accounts for partial training-set overlap between folds, which inflates
# the naive cross-fold variance estimate.
# Formula: SE = sqrt((1/(K*R) + N2_OVER_N1) * var)  with df = K*R - 1
# where N2_OVER_N1 = n_test / n_train = (1/K) / ((K-1)/K) = 1/(K-1)
BF_K          = N_FOLDS    # folds per repeat
BF_R          = N_REPEATS  # number of repeats
BF_N2_N1      = 1.0 / (BF_K - 1)   # = 0.25 for 5-fold CV
BF_DF         = BF_K * BF_R - 1    # = 19
BF_SCALE      = 1.0 / (BF_K * BF_R) + BF_N2_N1  # multiplier inside sqrt

STANDALONE_MODELS = {'CausalPFN': 'CausalPFN', 'CF': 'CausalForest'}

rows = []
for meta in META_LEARNERS:
    if meta in STANDALONE_MODELS:
        model_list = [STANDALONE_MODELS[meta]]
    else:
        model_list = BASE_MODELS
    for name in model_list:
        d = results[meta][name]
        if not d['auqc']:
            continue
        row = {'Meta': meta, 'Base': name}

        def agg(vals, label):
            arr = np.array(vals, dtype=float)  # shape (N_SPLITS,)
            valid_folds = arr[~np.isnan(arr)]
            nf = len(valid_folds)

            mean = np.nanmean(valid_folds) if nf > 0 else np.nan
            # Bouckaert-Frank corrected SE
            var          = np.nanvar(valid_folds, ddof=1) if nf > 1 else np.nan
            se_corrected = np.sqrt(BF_SCALE * var)        if nf > 1 else np.nan

            row[f'{label} Mean'] = mean
            row[f'{label} SE']   = se_corrected
            row[f'{label} N']    = nf

            if label == 'AUQC' and nf > 1:
                tcrit = st.t.ppf(0.975, df=BF_DF)
                ci_lo, ci_hi = mean - tcrit * se_corrected, mean + tcrit * se_corrected
                row[f'{label} CI']  = f"[{ci_lo:.4f}, {ci_hi:.4f}]"
                # One-sided: Y=aliveind6 is a positive outcome, so only flag positive AUQC
                row[f'{label} Sig'] = '*' if ci_lo > 0 else ''
            return row

        agg(d['auqc'], 'AUQC')
        for k in TOP_K_PERCENTAGES:
            agg(d[f'gain_{k}'], f'Gain@{k}%')

        rows.append(row)

df = pd.DataFrame(rows)

# Sort by AUQC Mean descending (best first: highest AUQC = best, Y=aliveind6 is a positive outcome)
df = df.sort_values('AUQC Mean', ascending=False).reset_index(drop=True)

# --- Display: AUQC ---
print("=" * 110)
print(f"RESULTS: {N_FOLDS} folds × {N_REPEATS} repeats = {N_SPLITS} splits")
print(f"Mean ± SE (Bouckaert-Frank corrected, df={BF_DF}) — ranked by AUQC")
print("=" * 110)
print("AUQC is diagonal-subtracted: expected value under random model is 0 by construction.")

auqc_cols = ['Meta', 'Base', 'AUQC Mean', 'AUQC SE', 'AUQC CI', 'AUQC Sig']
auqc_cols = [c for c in auqc_cols if c in df.columns]  # CI/Sig absent when N_SPLITS == 1
print("\n--- AUQC (diagonal-subtracted) ---")
print(df[auqc_cols].to_string(index=False, float_format='%.4f'))

# --- Display: Policy Gains ---
for k in TOP_K_PERCENTAGES:
    gain_cols = ['Meta', 'Base', f'Gain@{k}% Mean', f'Gain@{k}% SE']
    existing = [c for c in gain_cols if c in df.columns]
    df_k = df.sort_values(f'Gain@{k}% Mean', ascending=False).reset_index(drop=True)
    print(f"\n--- Policy Gain @{k}% (ranked) ---")
    print(df_k[existing].to_string(index=False, float_format='%.2f'))

# --- Bar charts ---
n_bars = 1 + len(TOP_K_PERCENTAGES)
fig, axes = plt.subplots(1, n_bars, figsize=(6 * n_bars, 6))

# AUQC bar chart
ax = axes[0]
plot_data = df.pivot(index='Base', columns='Meta', values='AUQC Mean')
plot_err = df.pivot(index='Base', columns='Meta', values='AUQC SE')
plot_data.plot(kind='bar', yerr=plot_err, ax=ax, capsize=3, rot=0)
ax.set_title('AUQC (diagonal-subtracted)', fontsize=10)
ax.set_ylabel('Mean ± SE')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=7, loc='best')

# Gain@k% bars
for i, k in enumerate(TOP_K_PERCENTAGES):
    ax = axes[i + 1]
    plot_data = df.pivot(index='Base', columns='Meta', values=f'Gain@{k}% Mean')
    plot_err = df.pivot(index='Base', columns='Meta', values=f'Gain@{k}% SE')
    plot_data.plot(kind='bar', yerr=plot_err, ax=ax, capsize=3, rot=0)
    ax.set_title(f'Gain@{k}% (policy − random)', fontsize=10)
    ax.set_ylabel('Mean ± SE')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=7, loc='best')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'bar_charts.pdf'), bbox_inches='tight')
plt.show()

# --- Qini Curves: 2x4 grid ---
x_grid = np.linspace(0, 1, N_GRID)
colors = {'LinearRegression': 'blue', 'LightGBM': 'green', 'TabPFN_v2.5': 'red', 'TabPFN_v2.6': 'orange',
          'TabICL': 'teal', 'CausalForest': 'brown', 'CausalPFN': 'purple'}

# --- Diagonal baseline (mean across splits, no SE band) ---
_g1_per_split = results['S']['LightGBM']['g1']
_all_diag = np.array([x_grid * g1 for g1 in _g1_per_split])
diag_mean = _all_diag.mean(axis=0)

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

legend_handles = []
legend_labels = []

for idx, meta in enumerate(['S', 'T', 'X', 'R', 'DR']):
    ax = axes[idx]
    for base in BASE_MODELS:
        curves_arr = qini_curves[meta][base]
        if curves_arr:
            c = np.array(curves_arr)
            m = c.mean(axis=0)
            s = c.std(axis=0) / np.sqrt(len(c))
            line, = ax.plot(x_grid, m, color=colors[base], linewidth=2)
            ax.fill_between(x_grid, m - s, m + s, color=colors[base], alpha=0.12)
            if idx == 0:
                legend_handles.append(line)
                legend_labels.append(base)
    diag_line, = ax.plot(x_grid, diag_mean, 'k--', linewidth=1, alpha=0.5)
    if idx == 0:
        legend_handles.append(diag_line)
        legend_labels.append('Diagonal (random)')
    ax.set_title(f'{meta}-Learner')
    ax.set_xlabel('Fraction targeted')
    ax.set_ylabel('Cumulative incremental outcome')
    ax.grid(True, alpha=0.3)

ax = axes[5]
curves_arr = qini_curves['CF']['CausalForest']
if curves_arr:
    c = np.array(curves_arr)
    m = c.mean(axis=0)
    s = c.std(axis=0) / np.sqrt(len(c))
    cf_line, = ax.plot(x_grid, m, color=colors['CausalForest'], linewidth=2)
    ax.fill_between(x_grid, m - s, m + s, color=colors['CausalForest'], alpha=0.12)
    legend_handles.append(cf_line)
    legend_labels.append('CausalForest')
ax.plot(x_grid, diag_mean, 'k--', linewidth=1, alpha=0.5)
ax.set_title('Causal Forest (DML)')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental outcome')
ax.grid(True, alpha=0.3)

ax = axes[6]
curves_arr = qini_curves['CausalPFN']['CausalPFN']
if curves_arr:
    c = np.array(curves_arr)
    m = c.mean(axis=0)
    s = c.std(axis=0) / np.sqrt(len(c))
    cpfn_line, = ax.plot(x_grid, m, color=colors['CausalPFN'], linewidth=2)
    ax.fill_between(x_grid, m - s, m + s, color=colors['CausalPFN'], alpha=0.12)
    legend_handles.append(cpfn_line)
    legend_labels.append('CausalPFN')
ax.plot(x_grid, diag_mean, 'k--', linewidth=1, alpha=0.5)
ax.set_title('CausalPFN')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental outcome')
ax.grid(True, alpha=0.3)

axes[7].set_visible(False)

fig.legend(legend_handles, legend_labels, loc='center', bbox_to_anchor=(0.875, 0.27),
           fontsize=9, frameon=True)

plt.suptitle(f'Qini Curves ({N_FOLDS}×{N_REPEATS} RSKF, mean ± 1 SE over splits)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'qini_curves.pdf'), bbox_inches='tight')
plt.show()